In [1]:
import pandas as pd
df=pd.read_csv("diabetes.csv")
print(df.shape)
print((df[["Glucose","BloodPressure","SkinThickness","Insulin","BMI"]]==0).sum())

(768, 9)
Glucose            5
BloodPressure     35
SkinThickness    227
Insulin          374
BMI               11
dtype: int64


In [2]:
import numpy as np
cols=["Glucose","BloodPressure","SkinThickness","Insulin","BMI"]
df[cols]=df[cols].replace(0,np.nan)
print(df[cols].isna().sum())

Glucose            5
BloodPressure     35
SkinThickness    227
Insulin          374
BMI               11
dtype: int64


In [4]:
from sklearn.model_selection import train_test_split
X=df.drop("Outcome",axis=1)
y=df["Outcome"]
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,stratify=y,random_state=42)
print(X_train.shape,X_test.shape)
print(y_train.mean(),y_test.mean())

(614, 8) (154, 8)
0.3485342019543974 0.35064935064935066


In [5]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold,cross_val_score

pipe=Pipeline([
    ("impute",SimpleImputer(strategy="median")),
    ("scale",StandardScaler()),
    ("model",LogisticRegression(max_iter=1000))
])
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
scores=cross_val_score(pipe,X_train,y_train,cv=cv,scoring="accuracy")
print(scores)
print(scores.mean(),scores.std())

[0.78861789 0.78861789 0.82113821 0.7804878  0.76229508]
0.7882313741170198 0.019056824693417025


In [6]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import classification_report
y_cv=cross_val_predict(pipe,X_train,y_train,cv=cv)
print(classification_report(y_train,y_cv))

              precision    recall  f1-score   support

           0       0.80      0.90      0.85       400
           1       0.76      0.57      0.65       214

    accuracy                           0.79       614
   macro avg       0.78      0.74      0.75       614
weighted avg       0.78      0.79      0.78       614



In [7]:
import numpy as np
from sklearn.metrics import precision_recall_curve,average_precision_score
proba=cross_val_predict(pipe,X_train,y_train,cv=cv,method="predict_proba")[:,1]
prec,rec,thr=precision_recall_curve(y_train,proba)
print(average_precision_score(y_train,proba))
i=np.where(rec[:-1]>=0.80)[0][-1]
print(thr[i],prec[i],rec[i])

0.723066927296932
0.28058829775410216 0.581081081081081 0.8037383177570093


In [8]:
from sklearn.metrics import confusion_matrix,classification_report
threshold=thr[i]
pipe.fit(X_train,y_train)
test_proba=pipe.predict_proba(X_test)[:,1]
for t in [0.5,threshold]:
    pred=(test_proba >=t).astype(int)
    print("threshold",round(t,3))
    print(confusion_matrix(y_test,pred))
    print(classification_report(y_test,pred))


threshold 0.5
[[82 18]
 [27 27]]
              precision    recall  f1-score   support

           0       0.75      0.82      0.78       100
           1       0.60      0.50      0.55        54

    accuracy                           0.71       154
   macro avg       0.68      0.66      0.67       154
weighted avg       0.70      0.71      0.70       154

threshold 0.281
[[67 33]
 [ 9 45]]
              precision    recall  f1-score   support

           0       0.88      0.67      0.76       100
           1       0.58      0.83      0.68        54

    accuracy                           0.73       154
   macro avg       0.73      0.75      0.72       154
weighted avg       0.77      0.73      0.73       154

